# 02 — Generator Pipelines: Unix Pipes in Python

**Goal:** Chain generators together like Unix pipes. This pattern is the foundation for async pipelines.

## The Unix pipe analogy

```bash
cat access.log | grep 'ERROR' | cut -d' ' -f1 | sort | uniq -c
```

Each command reads input lazily, processes it, outputs to the next. No command waits for the previous one to finish. Data FLOWS through.

Generators do exactly this:
```python
pipeline = upper(extract_message(grep("ERROR", generate_lines())))
for result in pipeline:
    print(result)
```

Each generator takes another iterable as input and yields transformed items.

## Exercise 2.1 — A simple pipeline

Build four generator functions:
1. `generate_lines()` — yields a list of log lines (mix of ERROR, INFO, WARNING)
2. `grep(pattern, lines)` — yields only lines containing `pattern`
3. `extract_message(lines)` — yields the part after `": "` in each line
4. `upper(lines)` — yields each line uppercased

Chain them: `upper(extract_message(grep("ERROR", generate_lines())))` and iterate.

In [ ]:
# Exercise 2.1 — Build and chain the pipeline


### Question 2.1
How many lines are in memory at ANY point during this pipeline?

**Key insight:** At most ONE line flows through the pipeline at a time. Each stage processes a single item and passes it to the next. This is how async data flows work too.

*Your answer:*


## Exercise 2.2 — Visualize the flow

Build three generators that print when they receive/produce items:
1. `source()` — yields 0, 1, 2, 3 with a print before each yield
2. `double(nums)` — yields `n * 2` for each `n`, printing what it got and yields
3. `add_ten(nums)` — yields `n + 10` for each `n`, printing what it got and yields

Chain them: `add_ten(double(source()))`. Print the final values.

**Watch the order.** Does `source` produce ALL values first, then `double` processes them? Or does each value flow through the ENTIRE pipeline before the next starts?

In [ ]:
# Exercise 2.2 — Visualize flow through the pipeline


### Question 2.2
Is this **push-based** or **pull-based** processing? Who drives the pipeline — the producer or the consumer?

*Your answer:*


## Exercise 2.3 — Build a log analyzer pipeline

First, run the cell below to create a fake log file. Then build the pipeline.

In [ ]:
import random
from collections import Counter

# Create fake log file
errors = ["disk full", "connection timeout", "file not found", "disk full", "auth failed"]
levels = ["INFO", "WARNING", "ERROR"]
with open('/tmp/app.log', 'w') as f:
    for i in range(1000):
        level = random.choice(levels)
        msg = random.choice(errors) if level == "ERROR" else f"routine operation {i}"
        f.write(f"2024-01-{(i%28)+1:02d} {level}: {msg}\n")
print("Created /tmp/app.log (1000 lines)")

Build a pipeline with these stages:
1. `read_log(filename)` — yields raw lines from file
2. `parse_log(lines)` — yields `(date, level, message)` tuples. Line format: `"2024-01-15 ERROR: disk full"`
3. `filter_level(records, level)` — yields only records matching the given level
4. `count_messages(records)` — returns a `Counter` of message occurrences (NOT a generator — it needs all data)

Chain them and print the top 5 errors.

In [ ]:
# Exercise 2.3 — Build the log analyzer pipeline


### Question 2.3
Why can't `count_messages` be a generator (lazy)? What's the fundamental difference between filter/transform stages and aggregation stages?

**This matters in async too:** Some operations can stream (lazy), some must buffer everything (eager). Knowing which is which is key to designing concurrent systems.

*Your answer:*


## Mastery Test: Lazy CSV Processor

Run the cell below to create test data, then build the pipeline.

In [ ]:
import csv

with open('/tmp/sales.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['date', 'product', 'quantity', 'price', 'region'])
    products = ['Widget', 'Gadget', 'Doohickey']
    regions = ['North', 'South', 'East', 'West']
    for i in range(10000):
        writer.writerow([
            f"2024-{(i%12)+1:02d}-{(i%28)+1:02d}",
            random.choice(products), random.randint(1, 100),
            round(random.uniform(5, 500), 2), random.choice(regions)
        ])
print("Created /tmp/sales.csv (10,000 rows)")

Build a generator pipeline that processes a CSV with constant memory:
1. `read_csv(filename)` — yields rows as dicts (use `csv.DictReader`)
2. `filter_rows(rows, column, value)` — yields only rows where `row[column] == value`
3. `select_columns(rows, *columns)` — yields rows with only the specified columns
4. `to_tsv(rows)` — yields tab-separated strings

Chain: `to_tsv(select_columns(filter_rows(read_csv(...), 'region', 'North'), 'date', 'product', 'quantity'))`

Use your `take()` from notebook 01 to print just the first 10 results.

In [ ]:
# Mastery Test — Build the lazy CSV pipeline


## Summary

```
source() | filter() | transform() | sink()
   ↓          ↓           ↓          ↓
  yield →   yield →     yield →    consume
```

One item flows through the ENTIRE pipeline before the next starts. This is PULL-based: the consumer drives everything.

**Connection to async:**
- Generator pipeline = pull-based (consumer calls `next()`)
- Async pipeline = push-based (producer `put()`s into queue)
- Next module shows how `yield` + `.send()` flips generators from pull to push — turning them into coroutines

**Next module:** Module 01 — Generators as Coroutines (`.send()`, `yield from`, building a scheduler)